# 04 감성 — KNU 한국어 감성사전 (핵심의제 중심, 길이정규화·부정어 반전·0매칭 가드)

전처리본 `body_cleaned`를 Kiwi로 분석해 KNU 사전과 매칭, 매체그룹별 정서값 비교

- 입력 한 방식 — `body_cleaned`를 Kiwi로 **명사+형용사(VA)+동사(VV)+부정(`않다/없다/못하다`)** 추출 → KNU 표제어·어근 매칭. 원문 직접매칭과 안 섞음
- **부정어가 나타나면 그 직전 1~2 형태소 polarity 부호 반전**(부정표지가 평가어 뒤에 옴)
- **점수 = 매칭 토큰 polarity 평균(합산 아님 — 길이효과 제거)**. `matched_count==0`이면 **NaN**(0/0 방지)·그룹평균서 제외, **미검출 비율 보고**
- 대상 = **핵심 의제 기사 중심**(02/03의 핵심의제). 한계 — KNU는 매체 논조가 아니라 등장 사건 어휘 정서값(룰기반)


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, json, unicodedata, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager as fm

try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists(): raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR)
RESULT_DIR = PROJECT_DIR / 'result'; DATA_DIR = PROJECT_DIR / 'news'
GROUPS = ['경제','통신·보도','정치색','지상파']
for cand in ['NanumGothic','Malgun Gothic','AppleGothic']:
    if any(cand.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = cand; break
plt.rcParams['axes.unicode_minus'] = False
def nn(p): return unicodedata.normalize('NFC', p.name)


In [ ]:
# --- KNU 감성사전 로드(선행 확보 필수) ---
# park1200656/KnuSentiLex 의 SentiWord_info.json 을 resources/knu_sentiment.json 로 저장해둘 것
cands = [PROJECT_DIR/'resources'/'knu_sentiment.json', PROJECT_DIR/'resources'/'SentiWord_info.json']
KNU_PATH = next((p for p in cands if p.exists()), None)
if KNU_PATH is None:
    raise FileNotFoundError('KNU 사전 없음 — park1200656/KnuSentiLex의 SentiWord_info.json을 resources/knu_sentiment.json로 저장 후 재실행')
raw = json.loads(KNU_PATH.read_text(encoding='utf-8'))
# 재현성 — 사전 체크섬 기록
print('KNU 파일:', KNU_PATH.name, '/ 표제어', len(raw), '/ md5', hashlib.md5(KNU_PATH.read_bytes()).hexdigest())

# 스키마 word/word_root/polarity → 표제어·어근 둘 다 dict(키 충돌 시 절댓값 큰 극성 우선)
KNU = {}
def put(key, pol):
    key = str(key).strip()
    if not key: return
    if key not in KNU or abs(pol) > abs(KNU[key]): KNU[key] = pol
for e in raw:
    try: pol = int(e.get('polarity'))
    except Exception: continue
    put(e.get('word',''), pol)
    if e.get('word_root'): put(e.get('word_root'), pol)
print('KNU 매칭 키:', len(KNU), '/ 예:', list(KNU.items())[:3])


In [ ]:
# --- 분석 대상 로드 — 핵심의제 문서 + body_cleaned join ---
def find_one(pat, base=RESULT_DIR, period=None):
    cc = sorted(p for p in base.iterdir() if p.is_file() and re.match(pat, nn(p)) and (period is None or period in nn(p)))
    if not cc: raise FileNotFoundError(pat + (f' (기간 {period})' if period else '') + ' 없음 — 앞 단계 먼저')
    return cc[-1]
DT_PATH = find_one(r'^문서토픽분포_언론사_\d{6}_\d{6}\.csv$')
PERIOD = re.search(r'(\d{6}_\d{6})', nn(DT_PATH)).group(1)
# ★ PRE/핵심의제는 DT와 동일 PERIOD로 강제 — 다기간 혼선·엉뚱한 본문 join 방지(codex)
PRE_PATH = find_one(r'^전처리_본문_언론사_\d{6}_\d{6}\.csv$', DATA_DIR, period=PERIOD)
CORE_PATH = RESULT_DIR / f'핵심의제_라벨_{PERIOD}.csv'
if not CORE_PATH.exists():
    raise FileNotFoundError('핵심의제_라벨_*.csv 없음 — 03에서 CORE_TOPICS·라벨 확정·저장 후 재실행(03↔04 타깃 일치)')

dt = pd.read_csv(DT_PATH, encoding='utf-8-sig')
# ★ 핵심의제는 03에서 사람이 확정·저장한 라벨 파일을 그대로 읽음(03과 타깃 일치, 자동 top4 금지)
CORE_TOPICS = pd.read_csv(CORE_PATH, encoding='utf-8-sig')['topic'].tolist()
print('핵심의제(03 확정):', CORE_TOPICS)
body = pd.read_csv(PRE_PATH, encoding='utf-8-sig', usecols=['article_id','body_cleaned'])
df = dt.merge(body, on='article_id', how='left', validate='one_to_one')
assert df['body_cleaned'].notna().all(), 'body_cleaned join 결측 — DT와 전처리본 기간/키 불일치'
df['high_purity'] = df['high_purity'].map({'True':True,'False':False,True:True,False:False})  # 문자열 직렬화 대비 bool 강제
# 핵심 의제 기사 중심 — 고순도 hard 문서 & dominant in 핵심의제(전체 평균은 의제믹스에 끌리므로)
core = df[df['high_purity'] & df['dominant_topic'].isin(CORE_TOPICS)].copy()
print('핵심의제 대상 문서:', len(core), '/ 그룹', core['media_group'].value_counts().to_dict())


In [ ]:
# --- Kiwi 형태소 → KNU 매칭(불규칙·파생 재구성, 부정어 후행 반전) ---
from kiwipiepy import Kiwi
kiwi = Kiwi()
CONTENT = {'NNG','NNP','VA','VV'}                    # 감성 후보 품사(접두 비교 — 불규칙 VA-I/VV-I 포함)
DERIV = {'XSA','XSV'}                                 # 파생접미사 하/되/스럽/롭 …
NEG_FORMS = {'않','없','못','못하'}                    # 않다(않)/없다(없)/못하다(못하·VX)/못(MAG) — spec 한정
def bt(tok): return tok.tag.split('-')[0]            # 불규칙 태그(VA-I 등) 접두만
def is_neg(tok): return (tok.form in NEG_FORMS) and (tok.tag[0] in {'V','M'})

def doc_sentiment(text):
    toks = kiwi.tokenize(str(text)); n = len(toks)
    pols = []; i = 0
    while i < n:
        b = bt(toks[i]); cands = None; end = i
        if b in {'NNG','NNP'}:
            if i+1 < n and bt(toks[i+1]) in DERIV:        # 명사+파생 → 불안+하 = 불안하다, 안정+되 = 안정되다, 자랑+스럽 = 자랑스럽다
                cands = [toks[i].form + toks[i+1].form + '다', toks[i].form]; end = i+1
            else:
                cands = [toks[i].form]
        elif b in {'VA','VV'}:                            # 용언 어간+'다' (불규칙 포함)
            cands = [toks[i].form + '다', toks[i].form]
        if cands:
            pol = next((KNU[c] for c in cands if c in KNU), None)
            if pol is not None:
                # 평가어 직후 1~2 '비어미' 형태소에 부정어 있으면 반전(어미 지/어 등은 건너뜀)
                cnt = 0; j = end + 1; neg = False
                while j < n and cnt < 2:
                    if toks[j].tag[0] == 'E': j += 1; continue
                    if is_neg(toks[j]): neg = True; break
                    cnt += 1; j += 1
                pols.append(-pol if neg else pol)
        i = end + 1
    if not pols:
        return np.nan, 0
    return float(np.mean(pols)), len(pols)        # ★ 평균(길이정규화), 매칭 0이면 NaN

print('샘플:', doc_sentiment('정부의 대응은 훌륭했지만 시장 불안은 해소되지 않았다'))


In [ ]:
# --- 전체 점수 + 그룹별 집계(0매칭 가드) ---
try:
    from tqdm.auto import tqdm; tqdm.pandas(desc='감성')
    res = core['body_cleaned'].progress_map(doc_sentiment)
except Exception:
    res = core['body_cleaned'].map(doc_sentiment)
core['senti'] = [r[0] for r in res]
core['matched'] = [r[1] for r in res]

# 라벨 — 임계 토글(기본 ±0.1). NaN(미검출)은 분류서 제외
THR = 0.1
def lab(s):
    if pd.isna(s): return 'na'
    return 'pos' if s > THR else ('neg' if s < -THR else 'neu')
core['senti_label'] = core['senti'].map(lab)

# ★ 미검출(matched==0)은 NaN → 그룹 평균서 제외, 미검출 비율 별도 보고
cov = core.groupby('media_group')['matched'].apply(lambda s: (s>0).mean()*100).round(1).reindex(GROUPS)
mean_senti = core[core['matched']>0].groupby('media_group')['senti'].mean().round(3).reindex(GROUPS)
valid = core[core['matched']>0]
ratio = (valid.groupby('media_group')['senti_label'].value_counts(normalize=True)
           .unstack(fill_value=0).reindex(GROUPS).reindex(columns=['pos','neu','neg'], fill_value=0).round(3))
print('그룹별 감성 검출(커버리지) 비율 %:'); print(cov)
print('그룹별 평균 polarity(미검출 제외):'); print(mean_senti)
print('그룹별 긍/중/부 비율(미검출 제외):'); print(ratio)


In [ ]:
# --- 막대그래프 + 저장 ---
fig, ax = plt.subplots(1, 2, figsize=(11,4))
mean_senti.plot(kind='bar', ax=ax[0], color='steelblue'); ax[0].axhline(0, color='k', lw=.6)
ax[0].set_title('그룹별 평균 polarity(핵심의제·미검출 제외)'); ax[0].set_ylabel('평균 polarity')
ratio.plot(kind='bar', stacked=True, ax=ax[1], color=['#4c72b0','#bbbbbb','#c44e52'])
ax[1].set_title('그룹별 긍/중/부 비율'); ax[1].legend(loc='lower right')
for a in ax: a.set_xticklabels(a.get_xticklabels(), rotation=0)
fig.tight_layout(); fig.savefig(RESULT_DIR / f'감성_막대_{PERIOD}.png', dpi=150)

summary = pd.DataFrame({'커버리지%': cov, '미검출%': (100-cov).round(1), '평균polarity': mean_senti})
summary = summary.join(ratio)
summary.to_csv(RESULT_DIR / f'감성요약_언론사_{PERIOD}.csv', encoding='utf-8-sig')
core[['article_id','media_group','dominant_topic','senti','matched','senti_label']].to_csv(
    RESULT_DIR / f'감성_문서별_{PERIOD}.csv', index=False, encoding='utf-8-sig')
print('저장: 감성요약 / 감성_문서별 / 감성_막대.png')
print('한계 — KNU는 매체 논조가 아니라 등장 사건 어휘의 정서값(룰기반·문맥 부분반영). 미검출률 함께 해석')


## 검증 체크리스트
- KNU 사전 선행 확보(없으면 멈춤)·표제어 수·md5 기록
- 입력 한 방식(body_cleaned·Kiwi 명사+VA+VV+부정), 원문 직접매칭과 안 섞음
- 부정어 직전 1~2 형태소 반전 적용
- 점수=매칭 polarity 평균(합산 아님), matched==0 → NaN·그룹평균 제외, 미검출 비율 표 보고
- 핵심의제 기사 중심, 그룹 긍/중/부 비율 합 100%(미검출 제외 기준)
- 그래프 한글 정상
